# 🤖 Lesson 03: AutoGen Basics

Now we stop building agent loops manually and use **Microsoft AutoGen** — the framework that handles everything for us!

In [ ]:
# !pip install pyautogen python-dotenv rich
import os
from dotenv import load_dotenv
from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager
from rich.console import Console
load_dotenv()
console = Console()

# AutoGen uses a config list for LLM settings
llm_config = {
    'config_list': [{
        'model': os.getenv('AZURE_OPENAI_DEPLOYMENT', 'gpt-4o'),
        'api_type': 'azure',
        'api_key': os.getenv('AZURE_OPENAI_API_KEY'),
        'base_url': os.getenv('AZURE_OPENAI_ENDPOINT'),
        'api_version': os.getenv('AZURE_OPENAI_API_VERSION'),
    }],
    'temperature': 0
}
console.print('[green]✅ AutoGen configured with Azure OpenAI[/green]')

## 🤖 Part 1: Your First AutoGen Agent

In [ ]:
# AssistantAgent = AI agent (uses LLM)
dsa_tutor = AssistantAgent(
    name='DSA_Tutor',
    system_message="""You are an expert DSA tutor helping students prepare for interviews.
Explain concepts clearly, give time/space complexity, and provide Python code examples.
When done, say TERMINATE.""",
    llm_config=llm_config
)

# UserProxyAgent = represents the human (can execute code!)
student = UserProxyAgent(
    name='Student',
    human_input_mode='NEVER',  # Fully automated
    max_consecutive_auto_reply=3,
    code_execution_config={'work_dir': './code_output', 'use_docker': False},
    is_termination_msg=lambda x: 'TERMINATE' in x.get('content', '')
)

# Start the conversation!
student.initiate_chat(
    dsa_tutor,
    message='Explain the Two Sum problem and solve it in Python with O(n) time complexity.'
)

## 🎭 Part 2: Multi-Agent Group Chat

In [ ]:
# Create specialized agents
explainer = AssistantAgent(
    name='Explainer',
    system_message='You explain DSA concepts in simple terms. Focus on intuition first.',
    llm_config=llm_config
)

coder = AssistantAgent(
    name='Coder',
    system_message='You write clean, optimal Python code with comments. Include time/space complexity.',
    llm_config=llm_config
)

critic = AssistantAgent(
    name='Critic',
    system_message='You review explanations and code. Point out edge cases and suggest improvements. End with TERMINATE when satisfied.',
    llm_config=llm_config
)

user = UserProxyAgent(
    name='User',
    human_input_mode='NEVER',
    max_consecutive_auto_reply=1,
    is_termination_msg=lambda x: 'TERMINATE' in x.get('content', '')
)

# Group chat — agents take turns!
groupchat = GroupChat(
    agents=[user, explainer, coder, critic],
    messages=[],
    max_round=6
)

manager = GroupChatManager(groupchat=groupchat, llm_config=llm_config)

user.initiate_chat(
    manager,
    message='Teach me about Binary Search Trees: explain, code it, and review edge cases.'
)

## ✅ What AutoGen Gives Us
- No manual Perceive→Reason→Act loop needed
- Built-in code execution sandbox
- Multi-agent collaboration out of the box
- Azure OpenAI integration with one config